# Generalized Linear Models: Quantifying Single-Neuron Selectivity

### NEUBEH/PBIO 545 — Quantitative Methods in Neuroscience

*Adapted from* the DMTS GLM pipeline in `GlmnetDmtsDemo`.

Every other tutorial in this course that involves a decoder runs in the **decoding**
direction: given neural activity, what was the stimulus? `ClassificationTutorial` is entirely
about that. This tutorial runs the other way — the **encoding** direction. Given what the
animal saw and did on each frame of each trial, what does this neuron's firing rate do?

That reversal is not cosmetic. It is how the question *"is this neuron selective for X?"*
actually gets answered in practice, and it forces a problem that decoding largely hides:

> In a behaving animal, the variable you care about is correlated with a dozen variables you
> do not. If mice run faster on cue-1 trials, then a neuron that only encodes running speed
> will look beautifully cue-selective to any analysis that ignores speed.

A GLM lets you ask whether a task variable explains activity *over and above* the nuisance
variables — and that question has a quantitative answer, not a yes/no.

| Part | Topic |
|---|---|
| I | The Poisson GLM: log link, likelihood, and deviance |
| II | Fitting by IRLS, from scratch and then checked against statsmodels |
| III | Building a design matrix out of trial structure |
| IV | Simulating a population with known selectivity |
| V | Cross-validation by trial, and fraction of deviance explained |
| VI | Regularization: ridge, lasso, and what elastic net buys you |
| VII | **Nested model comparison — the selectivity measure** |
| VIII | Significance by trial shuffling, and a population summary |

**Prerequisites:** `LinearAlgebra`, `ClassificationTutorial` (which introduces logistic
regression, cross-validation and regularization — all of which reappear here).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

rng = np.random.default_rng(11)     # fixed seed, so every run reproduces the figures

plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
    "axes.titlesize": 11, "font.size": 9,
})

BLUE, RED, GREEN, PURPLE, ORANGE, GREY = (
    "#3366d9", "#d94d3f", "#33914f", "#6a4fa8", "#e08b1e", "#808080")
EPOCH_COLORS = {"iti": "#eeeeee", "sample": "#cfe2ff", "delay": "#ffe8cc",
                "test": "#d8f0d8", "reward": "#f3d9e8"}

---
## Part 0. The task, and what we are trying to measure

The design here follows a delayed match-to-sample (DMTS) experiment run in virtual reality.
A mouse runs down a 400 cm virtual T-maze. A **sample cue** appears, then disappears at the
start of a **delay** segment; at the end of the delay a **test cue** appears at a fixed
spatial location, and the mouse turns left or right depending on whether sample and test
match. Neural activity is two-photon calcium imaging, so we get one frame every ~67 ms.

We keep the *structure* of that task and simulate the data, for one specific reason: with
simulated data we know the right answer. We will plant neurons with known selectivity, and
then check whether the analysis recovers it — including one neuron deliberately built to
fool a naive analysis.

Each simulated trial runs through five periods:

| Period | What happens |
|---|---|
| ITI | inter-trial interval; nothing task-related |
| sample | the sample cue is visible (one of 4 cue positions) |
| delay | cue gone; the animal must hold it in memory |
| test | the test cue appears; the choice is made |
| reward | outcome delivered, correct or error |

**And one crucial detail, which is true of real data:** the animal's running speed differs
systematically between epochs *and* between cue conditions. That is the confound this whole
tutorial is built around.

In [ ]:
n_trials = 240
fps      = 15.0                     # imaging frame rate (Hz)
dt       = 1.0 / fps

EPOCHS = ["iti", "sample", "delay", "test", "reward"]

# Epoch durations in frames: (min, max), drawn per trial
DUR = {"iti": (8, 14), "sample": (12, 20), "delay": (15, 45),
       "test": (10, 16), "reward": (12, 20)}

# Mean running speed in each epoch (cm/s): fast down the delay, slowing into the test
SPEED_BY_EPOCH = {"iti": 3.0, "sample": 20.0, "delay": 34.0, "test": 14.0, "reward": 5.0}

# THE CONFOUND: the animal runs faster on cue-1 trials and slower on cue-4 trials.
# This is not an artifact we invented for the tutorial -- animals routinely run
# differently on different trial types, and it is exactly what makes "this neuron
# is cue-selective" such an easy claim to get wrong.
SPEED_BY_CUE = {1: +10.0, 2: 0.0, 3: 0.0, 4: -5.0}

cue     = rng.integers(1, 5, n_trials)              # cue position, 1..4
correct = rng.random(n_trials) < 0.75               # 75% correct

def smooth_noise(n, tau_frames, rng):
    '''Temporally correlated noise: white noise through an exponential kernel.'''
    k = np.exp(-np.arange(0, 6 * tau_frames) / tau_frames)
    k /= k.sum()
    return np.convolve(rng.standard_normal(n + len(k)), k, mode="same")[:n]

# --- build the frame-by-frame session ---
rows = []           # one entry per frame: (trial, epoch, cue, correct, speed)
for t in range(n_trials):
    for ep in EPOCHS:
        n_f = rng.integers(*DUR[ep])
        base = SPEED_BY_EPOCH[ep] + (SPEED_BY_CUE[cue[t]] if ep != "iti" else 0.0)
        sp = base + 6.0 * smooth_noise(n_f, 3.0, rng)
        for f in range(n_f):
            rows.append((t, ep, cue[t], correct[t], max(sp[f], 0.0)))

trial_id  = np.array([r[0] for r in rows])
epoch     = np.array([r[1] for r in rows])
cue_frame = np.array([r[2] for r in rows])
corr_frame= np.array([r[3] for r in rows])
speed     = np.array([r[4] for r in rows])
n_frames  = len(rows)

print(f"{n_trials} trials, {n_frames} frames "
      f"({n_frames*dt:.0f} s = {n_frames*dt/60:.1f} min of imaging)")
print(f"mean speed by epoch: " +
      ", ".join(f"{e} {speed[epoch==e].mean():.1f}" for e in EPOCHS) + " cm/s")
print(f"mean speed by cue  : " +
      ", ".join(f"cue{c} {speed[cue_frame==c].mean():.1f}" for c in (1,2,3,4)) + " cm/s")

In [ ]:
# Look at a few trials, and at the confound
fig, axes = plt.subplots(2, 1, figsize=(11, 5.5),
                         gridspec_kw={"height_ratios": [2, 1.1]})

show = np.isin(trial_id, np.arange(4))
ax = axes[0]
ax.plot(np.arange(show.sum()) * dt, speed[show], "k", lw=1.2)
# shade the epochs
start = 0
for i in range(1, show.sum() + 1):
    if i == show.sum() or epoch[show][i] != epoch[show][start]:
        ax.axvspan(start*dt, i*dt, color=EPOCH_COLORS[epoch[show][start]], zorder=0)
        start = i
for t in np.arange(4):
    seg = trial_id[show] == t
    ax.text(np.flatnonzero(seg)[0]*dt + 0.3, 58, f"trial {t}  cue {cue[t]}", fontsize=8)
ax.set(xlabel="Time (s)", ylabel="Running speed (cm/s)", ylim=(0, 65),
       title="Four example trials (shading = epoch)")
ax.legend(handles=[mpatches.Patch(color=EPOCH_COLORS[e], label=e) for e in EPOCHS],
          ncol=5, fontsize=8, loc="lower right")

ax = axes[1]
for c, col in zip((1, 2, 3, 4), (RED, BLUE, GREEN, PURPLE)):
    m = [speed[(cue_frame == c) & (epoch == e)].mean() for e in EPOCHS]
    ax.plot(range(len(EPOCHS)), m, "-o", color=col, lw=1.8, label=f"cue {c}")
ax.set_xticks(range(len(EPOCHS))); ax.set_xticklabels(EPOCHS)
ax.set(ylabel="Mean speed (cm/s)",
       title="The confound: speed depends on epoch AND on cue")
ax.legend(fontsize=8, ncol=4)
fig.tight_layout()

The lower panel is the entire problem in one picture. Speed varies across epochs — that is
expected, the animal accelerates down the delay segment and slows to make its choice — but it
*also* varies across cue conditions. Any neuron whose firing tracks running speed will
therefore have cue-dependent activity, with no cue selectivity whatsoever.

---
## Part I. The Poisson GLM

A generalized linear model has three parts. Choosing them is the whole modelling decision.

**1. A linear predictor.** A weighted sum of covariates, exactly as in every linear method
in this course:

$$\eta_t = \beta_0 + \sum_j \beta_j x_{jt}$$

**2. A link function**, connecting the linear predictor to the mean of the response. For
count data we use the **log link**, $\log \mu_t = \eta_t$, so that

$$\mu_t = \exp(\eta_t)$$

Two reasons. The rate stays positive for any $\beta$, which a linear model cannot guarantee.
And effects become **multiplicative**: a coefficient of $\beta_j = 0.7$ means "this covariate
multiplies the rate by $e^{0.7} \approx 2$", which is how gain modulation actually works in
neurons.

**3. A noise model.** For spike counts in a bin, the Poisson distribution:

$$P(y_t \mid \mu_t) = \frac{\mu_t^{y_t} e^{-\mu_t}}{y_t!}$$

Its log-likelihood, dropping the $y!$ term that does not depend on $\beta$, is

$$\ell(\beta) = \sum_t \left[ y_t \eta_t - e^{\eta_t} \right]$$

### Deviance is the currency

Rather than the likelihood itself, GLM work is done in **deviance**: twice the log-likelihood
gap between your model and a perfect ("saturated") model that predicts every observation
exactly.

$$D = 2\sum_t \left[ y_t \log\frac{y_t}{\mu_t} - (y_t - \mu_t) \right]$$

with the convention $y\log y = 0$ when $y = 0$. Deviance is the GLM generalization of the
residual sum of squares — for a Gaussian model with identity link it *is* the residual sum of
squares. Lower is better, zero is a perfect fit.

Deviance matters here because it gives us a common yardstick. Define the **null model** as
intercept only: it predicts every frame with the mean rate. Then

$$\text{FDE} \;=\; \frac{D_{\text{null}} - D_{\text{model}}}{D_{\text{null}}}$$

the **fraction of deviance explained**, is the GLM analogue of $R^2$. Measured on *held-out*
trials it is the number we will use throughout.

In [ ]:
def poisson_deviance(y, mu, eps=1e-12):
    '''Poisson deviance, with the y log y -> 0 convention at y = 0.'''
    mu = np.maximum(mu, eps)
    term = np.where(y > 0, y * np.log(np.maximum(y, eps) / mu), 0.0)
    return 2.0 * np.sum(term - (y - mu))

def frac_dev_explained(y, mu_model, mu_null):
    '''FDE: how much of the null model's deviance this model removes.'''
    d_null = poisson_deviance(y, mu_null)
    return (d_null - poisson_deviance(y, mu_model)) / d_null

# Sanity checks -- worth doing whenever you implement a loss function.
y_demo = rng.poisson(3.0, 5000)
print(f"deviance of the TRUE mean          : {poisson_deviance(y_demo, np.full(5000, 3.0)):.1f}")
print(f"deviance of a perfect fit (mu = y) : {poisson_deviance(y_demo, np.maximum(y_demo,1e-12)):.2e}")
print(f"deviance of a bad guess (mu = 10)  : {poisson_deviance(y_demo, np.full(5000, 10.0)):.1f}")
print(f"FDE of the true mean vs itself     : {frac_dev_explained(y_demo, np.full(5000,3.0), np.full(5000, y_demo.mean())):.4f}")

The middle line is the definition working: when $\mu = y$ exactly, the deviance is zero. And
the last line is a useful reality check — the true mean *is* the null model here (the data
have no structure), so it explains essentially none of the null deviance. FDE near zero for a
model with nothing to find is the behavior we want.

---
## Part II. Fitting by IRLS

Maximizing the Poisson log-likelihood has no closed form, but the likelihood is concave, so
Newton's method converges quickly and reliably. Newton on a GLM is called **iteratively
reweighted least squares**, and the classification tutorial already used it for logistic
regression — only the weights and the mean function change.

For the Poisson with log link, the gradient and Hessian of the penalized log-likelihood are

$$\nabla = X^\top (y - \mu) - R\beta,
\qquad H = -X^\top \mathrm{diag}(\mu)\, X - R$$

where $R = \lambda I$ is a ridge penalty with the intercept left unpenalized. Note the
weights are just $\mu$ — for the Poisson, the variance equals the mean, so frames where the
neuron fires more get more weight.

In [ ]:
def poisson_glm_fit(X, y, lam=0.0, max_iter=200, tol=1e-10):
    '''Poisson GLM with log link and an optional ridge penalty, by IRLS.

    X must already contain a column of ones for the intercept, which is NOT
    penalized. Returns the coefficient vector.
    '''
    n, p = X.shape
    beta = np.zeros(p)
    beta[0] = np.log(max(y.mean(), 1e-3))       # sensible start: the mean rate
    R = lam * np.eye(p); R[0, 0] = 0.0
    for _ in range(max_iter):
        mu = np.exp(np.clip(X @ beta, -30, 30))
        grad = X.T @ (y - mu) - R @ beta
        H    = X.T @ (X * mu[:, None]) + R
        step = np.linalg.solve(H + 1e-10 * np.eye(p), grad)
        beta = beta + step
        if np.max(np.abs(step)) < tol:
            break
    return beta

def poisson_predict(X, beta):
    return np.exp(np.clip(X @ beta, -30, 30))

# --- verify against statsmodels on synthetic data with known coefficients ---
import statsmodels.api as sm

p_test = 8
X_test = np.column_stack([np.ones(4000), rng.standard_normal((4000, p_test))])
beta_true = np.concatenate([[0.4], rng.normal(0, 0.4, p_test)])
y_test = rng.poisson(poisson_predict(X_test, beta_true))

beta_ours = poisson_glm_fit(X_test, y_test, lam=0.0)
beta_sm   = sm.GLM(y_test, X_test, family=sm.families.Poisson()).fit().params

print(f"max |ours - statsmodels|     : {np.abs(beta_ours - beta_sm).max():.2e}")
print(f"max |ours - true beta|       : {np.abs(beta_ours - beta_true).max():.3f}  (estimation error)")
print(f"deviance, ours vs statsmodels: {poisson_deviance(y_test, poisson_predict(X_test, beta_ours)):.4f}"
      f" vs {sm.GLM(y_test, X_test, family=sm.families.Poisson()).fit().deviance:.4f}")

Our fifteen lines agree with statsmodels to numerical precision. The residual difference from
the *true* coefficients is estimation error from finite data, which is a different thing
entirely and does not shrink by writing better code — only by collecting more trials.

---
## Part III. Building the design matrix

This is where most of the thinking goes, and where most mistakes are made. The design matrix
$X$ has one row per **frame** and one column per **predictor**. We build four blocks.

**Cue × epoch indicators (16 columns).** A separate regressor for each combination of cue
position (1–4) and epoch (sample, delay, test, reward). A neuron that fires for cue 2 only
while it is being held in memory loads on `cue2_in_delay` and nothing else. Coding cue and
epoch jointly rather than as separate main effects is what lets selectivity be
epoch-specific, which is the point of a working-memory task.

**Outcome (1 column).** `correct_in_reward`, marking rewarded frames.

**Running speed (15 columns).** Speed does not enter linearly. We tile the speed range with
five **raised-cosine bumps**, so the model can learn an arbitrary smooth speed-response curve
rather than being forced into a straight line. Each bump also enters at three time lags
(0, 2 and 4 frames), because calcium indicators are slow and the neuron's response to running
is smeared in time. This is the same idea as `runspeed_conv4.m` in the original pipeline.

**Acceleration (1 column).**

Total: 33 predictors plus an intercept. The intercept is identifiable because the ITI frames
carry no task regressors at all — they are the baseline every other condition is measured
against. That is a design choice worth making deliberately: if every frame belonged to some
epoch, the epoch columns would sum to one and be perfectly collinear with the intercept.

In [ ]:
def rcos_basis(x, centers, width):
    '''Raised-cosine bumps: each is a smooth hump of support 2*width around its centre.'''
    B = np.zeros((len(x), len(centers)))
    for i, c in enumerate(centers):
        d = np.pi * (x - c) / width
        m = np.abs(d) < np.pi
        B[m, i] = 0.5 * (1.0 + np.cos(d[m]))
    return B

def lag(v, k):
    '''Shift a column forward in time by k frames, zero-padding the start.'''
    out = np.zeros_like(v)
    if k == 0: return v.copy()
    out[k:] = v[:-k]
    return out

# ---- block 1: cue x epoch ----
cols, labels = [], []
for ep in ["sample", "delay", "test", "reward"]:
    for c in (1, 2, 3, 4):
        cols.append(((epoch == ep) & (cue_frame == c)).astype(float))
        labels.append(f"cue{c}_in_{ep}")

# ---- block 2: outcome ----
cols.append(((epoch == "reward") & corr_frame).astype(float))
labels.append("correct_in_reward")

# ---- block 3: speed, nonlinear (5 bumps) x 3 lags ----
sp_centers = np.linspace(0, 50, 5)
sp_width   = sp_centers[1] - sp_centers[0]
SB = rcos_basis(speed, sp_centers, sp_width)
for li, L in enumerate((0, 2, 4)):
    for bi in range(SB.shape[1]):
        cols.append(lag(SB[:, bi], L))
        labels.append(f"speed_b{bi}_lag{L}")

# ---- block 4: acceleration ----
accel = np.gradient(speed) * fps
cols.append(accel / np.std(accel))
labels.append("accel")

X_pred = np.column_stack(cols)                 # predictors, no intercept yet
X_all  = np.column_stack([np.ones(n_frames), X_pred])
labels_all = ["intercept"] + labels

TASK_COLS  = np.array([i for i, l in enumerate(labels_all)
                       if l.startswith("cue") or l == "correct_in_reward"])
MOTOR_COLS = np.array([i for i, l in enumerate(labels_all)
                       if l.startswith("speed") or l == "accel"])

print(f"design matrix: {X_all.shape[0]} frames x {X_all.shape[1]} columns "
      f"(1 intercept + {X_pred.shape[1]} predictors)")
print(f"  task block : {len(TASK_COLS)} columns")
print(f"  motor block: {len(MOTOR_COLS)} columns")
print(f"  rank = {np.linalg.matrix_rank(X_all)}  -> full rank: {np.linalg.matrix_rank(X_all) == X_all.shape[1]}")

In [ ]:
fig = plt.figure(figsize=(12.5, 5.5))
gs = fig.add_gridspec(1, 3, width_ratios=[2.1, 1, 1])

# --- the design matrix itself, first few trials ---
ax = fig.add_subplot(gs[0])
show2 = np.isin(trial_id, np.arange(5))
ax.imshow(X_all[show2].T, aspect="auto", cmap="magma", interpolation="nearest")
ax.set_yticks(range(len(labels_all)))
ax.set_yticklabels(labels_all, fontsize=5)
ax.set(xlabel="Frame", title="Design matrix, first 5 trials")
ax.grid(False)

# --- the speed basis ---
ax = fig.add_subplot(gs[1])
sgrid = np.linspace(-2, 55, 400)
for bi, col in enumerate(rcos_basis(sgrid, sp_centers, sp_width).T):
    ax.plot(sgrid, col, lw=2)
ax.set(xlabel="Running speed (cm/s)", ylabel="Basis weight",
       title="Speed basis: 5 raised cosines")

# --- collinearity between blocks ---
ax = fig.add_subplot(gs[2])
Z = X_pred - X_pred.mean(0)
sd = Z.std(0); sd[sd == 0] = 1
Cm = (Z / sd).T @ (Z / sd) / len(Z)
im = ax.imshow(Cm, cmap="RdBu_r", vmin=-1, vmax=1)
ax.axhline(16.5, color="k", lw=0.8); ax.axvline(16.5, color="k", lw=0.8)
ax.set(title="Predictor correlations\n(task block | motor block)")
ax.grid(False)
fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()

off_block = np.abs(Cm[:17, 17:])
print(f"largest |correlation| between a task and a motor predictor: {off_block.max():.3f}")
print(f"  ...between {labels[np.unravel_index(off_block.argmax(), off_block.shape)[0]]!r}"
      f" and {labels[17 + np.unravel_index(off_block.argmax(), off_block.shape)[1]]!r}")

The right-hand panel is worth dwelling on. The block structure is visible: task predictors
correlate with each other, motor predictors correlate with each other — but the
**off-diagonal blocks are not empty**. Task and motor predictors are correlated, because
speed depends on epoch and cue. That correlation is exactly what makes attributing activity
to one or the other a real statistical problem rather than a bookkeeping exercise.

> ### Homework question 1
> **(a)** Why is the ITI period necessary for the intercept to be identifiable? What would
> `np.linalg.matrix_rank` report if every frame belonged to sample, delay, test or reward?
>
> **(b)** The speed basis uses five bumps. What happens to the fit as you use one bump
> (nearly linear) or fifty? Relate your answer to the bias-variance trade-off from the
> classification tutorial.
>
> **(c)** We entered speed at lags 0, 2 and 4 frames but never at *negative* lags. What
> would a negative lag mean physically, and when might you legitimately want one?
>
> **(d)** Suppose you added a `cue2` main-effect column, active on every frame of every
> cue-2 trial. Check whether the design matrix is still full rank, and explain.

---
## Part IV. Simulating neurons with known selectivity

Now we plant five neurons. Each is a Poisson process whose log rate is a weighted sum of the
same design matrix columns we just built — so we know the exact right answer, and can ask
whether the analysis recovers it.

| Neuron | Built to be | What a correct analysis should conclude |
|---|---|---|
| `delay_cue` | fires for cue 2 during the delay | task-selective, specifically cue × delay |
| `test_epoch` | fires during test, equally for all cues | task-modulated but **not** cue-selective |
| `speed_only` | tracks running speed, nothing else | **not** task-selective, despite looking it |
| `mixed` | cue 3 during test, plus speed | both |
| `silent` | constant baseline | neither |

`speed_only` is the important one. Because the animal runs faster on cue-1 trials, this
neuron will show robustly different activity across cue conditions. A naive analysis will
call it cue-selective. It is not.

In [ ]:
def make_neuron(spec, baseline_rate):
    '''Build a true coefficient vector from a dict of {column label: weight}.'''
    b = np.zeros(len(labels_all))
    b[0] = np.log(baseline_rate)
    for lab, w in spec.items():
        matches = [i for i, l in enumerate(labels_all) if l.startswith(lab)]
        if not matches:
            raise KeyError(f"no column matches {lab!r}")
        for i in matches:
            b[i] = w
    return b

NEURONS = {
    # cue 2 held in memory through the delay
    "delay_cue":  make_neuron({"cue2_in_delay": 1.6}, 1.2),
    # any cue, but only during the test epoch
    "test_epoch": make_neuron({"cue1_in_test": 1.3, "cue2_in_test": 1.3,
                               "cue3_in_test": 1.3, "cue4_in_test": 1.3}, 1.2),
    # pure running-speed drive: rises across the speed basis at lag 0
    "speed_only": make_neuron({"speed_b2_lag0": 0.7, "speed_b3_lag0": 1.3,
                               "speed_b4_lag0": 1.8}, 0.8),
    # cue 3 during test AND speed
    "mixed":      make_neuron({"cue3_in_test": 1.4,
                               "speed_b3_lag0": 0.8, "speed_b4_lag0": 1.1}, 1.0),
    # nothing
    "silent":     make_neuron({}, 1.5),
}

Y = {}
for name, b in NEURONS.items():
    Y[name] = rng.poisson(poisson_predict(X_all, b))
    print(f"{name:<12} mean {Y[name].mean():5.2f} counts/frame  "
          f"({Y[name].mean()*fps:5.1f} events/s)   max {Y[name].max()}")

In [ ]:
# The naive analysis: average activity per cue, within each epoch.
fig, axes = plt.subplots(1, 5, figsize=(15, 3.2), sharex=True)
for ax, (name, y) in zip(axes, Y.items()):
    for c, col in zip((1, 2, 3, 4), (RED, BLUE, GREEN, PURPLE)):
        m = [y[(cue_frame == c) & (epoch == e)].mean() for e in EPOCHS]
        ax.plot(range(len(EPOCHS)), m, "-o", color=col, lw=1.6, ms=4, label=f"cue {c}")
    ax.set_xticks(range(len(EPOCHS)))
    ax.set_xticklabels(EPOCHS, rotation=45, ha="right", fontsize=7)
    ax.set_title(name)
axes[0].set_ylabel("Mean counts / frame")
axes[-1].legend(fontsize=7)
fig.suptitle("The naive analysis: mean activity by cue and epoch", y=1.04)
fig.tight_layout()

# Quantify the naive claim for the trap neuron
from scipy import stats
print("Naive one-way ANOVA across the 4 cues, using DELAY frames only:")
for name, y in Y.items():
    groups = [y[(cue_frame == c) & (epoch == "delay")] for c in (1, 2, 3, 4)]
    F, p = stats.f_oneway(*groups)
    verdict = "cue-selective!" if p < 0.001 else ""
    print(f"  {name:<12} F = {F:7.2f}   p = {p:.2e}   {verdict}")

And there is the trap, sprung. The ANOVA declares `speed_only` cue-selective in the delay
with an overwhelming p-value — and it is completely wrong. That neuron has no cue term at
all; it responds to running speed, and the animal runs faster on cue-1 trials.

Notice also that no amount of extra data fixes this. The p-value gets *smaller* with more
trials, because the effect is real — the neuron genuinely does fire differently across cue
conditions. What is false is the *interpretation*. Statistical significance is not the
problem; the omitted variable is.

> ### Homework question 2
> **(a)** `test_epoch` also comes out significant in some epochs. Is that a false positive
> in the same sense as `speed_only`? Explain the difference between "task-modulated" and
> "cue-selective".
>
> **(b)** Set every entry of `SPEED_BY_CUE` to zero and re-run from Part 0. Which p-values
> change, and which do not?
>
> **(c)** A common patch is to regress activity on speed first and run the ANOVA on the
> residuals. Under what conditions does that give the same answer as the GLM below, and when
> does it fail?

---
## Part V. Cross-validation by trial, and held-out FDE

Two decisions have to be made before any number can be trusted.

**Fit on some trials, evaluate on others.** A GLM with 34 parameters fit to 20,000 frames
will always improve the in-sample deviance. Only held-out deviance means anything.

**Split by trial, not by frame.** Frames within a trial are nearly the same observation —
same cue, same epoch, smoothly varying speed, and a calcium indicator that blurs everything
over hundreds of milliseconds. The classification tutorial raised this in homework 7(c).

We are going to do something slightly unusual here and *test* whether it matters, rather than
asserting that it does. The answer is instructive, and not the one you might expect.

We also **stratify by cue**, so every fold contains all four conditions, following the
`crossvalind` loop over cue types in the original `makeDM.m`.

In [ ]:
def make_trial_folds(trial_cue, k, rng):
    '''Assign whole TRIALS to k folds, stratified by cue condition.'''
    fold = np.empty(len(trial_cue), int)
    for c in np.unique(trial_cue):
        idx = np.flatnonzero(trial_cue == c)
        idx = idx[rng.permutation(len(idx))]
        fold[idx] = np.arange(len(idx)) % k
    return fold

k_folds     = 5
trial_folds = make_trial_folds(cue, k_folds, rng)
frame_fold  = trial_folds[trial_id]         # broadcast the trial's fold to its frames

print("frames per fold:", [int((frame_fold == f).sum()) for f in range(k_folds)])
print("cue counts per fold:")
for f in range(k_folds):
    print(f"  fold {f}: " + ", ".join(
        f"cue{c} {int((cue[trial_folds==f]==c).sum())}" for c in (1,2,3,4)))

def cv_fde(y, X, col_subset=None, lam=1.0, folds=frame_fold):
    '''Held-out fraction of deviance explained, pooled across folds.

    col_subset selects which design-matrix columns the model may use; the
    intercept (column 0) is always included. Passing an empty subset gives the
    null (intercept-only) model.
    '''
    cols = np.array([0]) if col_subset is None or len(col_subset) == 0 \
           else np.unique(np.concatenate([[0], col_subset]))
    mu_hat  = np.empty(len(y))
    mu_null = np.empty(len(y))
    for f in np.unique(folds):
        te, tr = folds == f, folds != f
        beta = poisson_glm_fit(X[tr][:, cols], y[tr], lam=lam)
        mu_hat[te] = poisson_predict(X[te][:, cols], beta)
        mu_null[te] = y[tr].mean()          # null fit on the SAME training data
    return frac_dev_explained(y, mu_hat, mu_null), mu_hat

In [ ]:
frame_folds = rng.integers(0, k_folds, n_frames)      # random frames, ignoring trials

all_cols = np.arange(1, X_all.shape[1])
print(f"{'neuron':<12}{'by trial':>10}{'by frame':>10}{'difference':>12}")
print("-" * 44)
for name, y in Y.items():
    a, _ = cv_fde(y, X_all, all_cols, lam=1.0, folds=frame_fold)
    b, _ = cv_fde(y, X_all, all_cols, lam=1.0, folds=frame_folds)
    print(f"{name:<12}{a:>10.4f}{b:>10.4f}{b-a:>+12.4f}")

**The difference is negligible — and that is worth understanding rather than glossing over.**

Frame-wise splitting leaks when the model has the capacity to memorize something specific to
an individual trial. Here it does not: all 33 predictors are *shared across trials*. There is
no per-trial parameter, no spike-history term, no slow drift basis. `cue2_in_delay` means the
same thing on trial 7 as on trial 200, so seeing 40 frames of trial 7 in training tells the
model nothing about trial 7 in particular. (Verify this yourself: add strong temporal
autocorrelation to the activity and the gap still does not open.)

So why split by trial anyway? Three reasons, none of which is "the point estimate is
inflated":

1. **The moment you add trial-specific flexibility, it leaks badly** — spike-history filters,
   per-trial gain terms, slow session-drift bases, trial-history regressors. All of these are
   standard, and all of them turn frame-wise CV into self-prediction.
2. **Your effective sample size is trials, not frames.** 20,000 frames sounds like a lot;
   240 trials is the honest number, and it is what sets the error bar on any claim.
3. **It changes significance dramatically**, even when it barely changes the point estimate.
   Part VIII demonstrates that — it is where the frame-versus-trial distinction really bites.

Also note the negative FDE for `silent`. That is not a bug: the fitted model predicts held-out
data *worse* than the training mean, which is the right verdict for a neuron with nothing to
explain.

> ### Homework question 3
> **(a)** Construct a design where frame-wise splitting *does* inflate FDE, by adding one
> column per trial (a per-trial offset). Predict the result before running it.
>
> **(b)** Our folds are stratified by cue but not by correct/error. The original pipeline
> stratifies by both, using eight groups. When would that matter?
>
> **(c)** The null model is refit on each training set rather than computed once from all
> data. Why does that matter for an honest FDE?
>
> **(d)** Report the fold-to-fold standard deviation of FDE alongside the mean. How does it
> compare to the differences between neurons in Part VII?

---
## Part VI. Regularization

With 33 predictors, many of them correlated, unregularized fits are unstable: two collinear
speed-basis columns can take large equal-and-opposite coefficients that cancel. The remedy is
the same as in the classification tutorial, now applied to a GLM.

- **Ridge** (L2, $\lambda\|\beta\|^2$) shrinks all coefficients smoothly toward zero.
- **Lasso** (L1, $\lambda\|\beta\|_1$) drives coefficients to *exactly* zero, selecting a
  subset of predictors.
- **Elastic net** mixes them: $\lambda\left[\alpha\|\beta\|_1 + \tfrac{1-\alpha}{2}\|\beta\|^2\right]$

The original pipeline uses `glmnet` with $\alpha = 0.95$ — almost pure lasso, with a touch of
ridge. That choice is deliberate, and the glmnet vignette explains why: with strongly
correlated predictors, pure lasso behaves erratically, arbitrarily picking one of a
correlated group and discarding the rest, and the numerics degrade. A small ridge component
removes that degeneracy while keeping sparse, interpretable solutions.

In [ ]:
lams = np.logspace(-2, 3, 22)
fde_by_lam = {name: [cv_fde(y, X_all, np.arange(1, X_all.shape[1]), lam=l)[0] for l in lams]
              for name, y in Y.items()}

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))

for (name, v), col in zip(fde_by_lam.items(), (BLUE, GREEN, RED, PURPLE, GREY)):
    axes[0].semilogx(lams, v, "-o", ms=3, lw=1.8, color=col, label=name)
axes[0].axhline(0, color="k", ls=":", lw=1)
axes[0].set(xlabel="Ridge penalty λ", ylabel="Held-out FDE",
            title="Cross-validated FDE vs regularization")
axes[0].legend(fontsize=8)

# ridge coefficient path for the mixed neuron
path = np.array([poisson_glm_fit(X_all, Y["mixed"], lam=l)[1:] for l in lams])
for j in range(path.shape[1]):
    is_task = (j + 1) in TASK_COLS
    axes[1].semilogx(lams, path[:, j], lw=1.2,
                     color=(BLUE if is_task else ORANGE), alpha=0.75)
axes[1].axhline(0, color="k", ls=":", lw=1)
axes[1].set(xlabel="Ridge penalty λ", ylabel="Coefficient",
            title="Ridge path, 'mixed' neuron (blue = task, orange = motor)")
fig.tight_layout()

best = {n: lams[int(np.argmax(v))] for n, v in fde_by_lam.items()}
print("λ maximizing held-out FDE, per neuron:")
for n, l in best.items():
    print(f"  {n:<12} λ = {l:8.2f}   FDE = {max(fde_by_lam[n]):+.4f}")

In [ ]:
# Elastic net, as glmnet does it: alpha = 0.95, lambda by cross-validation.
# statsmodels gives exact zeros, so we can see which predictors survive.
def elastic_net_fit(X, y, alpha_l1=0.95, pen=0.05):
    return sm.GLM(y, X, family=sm.families.Poisson()).fit_regularized(
        alpha=pen, L1_wt=alpha_l1).params

pens = np.logspace(-3, 0, 14)
rows = []
for pen in pens:
    b = elastic_net_fit(X_all, Y["mixed"], 0.95, pen)
    nz = np.abs(b[1:]) > 1e-8
    rows.append((pen, nz.sum(),
                 sum(nz[np.array(TASK_COLS) - 1]), sum(nz[np.array(MOTOR_COLS) - 1])))

print(f"{'penalty':>9} {'nonzero':>8} {'task':>6} {'motor':>6}")
for r in rows:
    print(f"{r[0]:>9.4f} {r[1]:>8} {r[2]:>6} {r[3]:>6}")

b_en = elastic_net_fit(X_all, Y["mixed"], 0.95, 0.05)
kept = [labels_all[i] for i in np.flatnonzero(np.abs(b_en) > 1e-8) if i > 0]
print(f"\nAt penalty 0.05 the elastic net keeps: {kept}")
print(f"True nonzero terms for 'mixed'        : "
      f"{[labels_all[i] for i in np.flatnonzero(NEURONS['mixed']) if i > 0]}")
print(f"\n'cue1_in_delay' is a FALSE POSITIVE -- it is 0.90 correlated with "
      f"speed_b4_lag0,\nso the penalty can trade one for the other at almost no cost "
      f"in likelihood.")

The elastic net recovers a sparse model containing the true cue-3-in-test term and speed
terms, discarding most of the 33 predictors. That sparsity is the practical appeal: the
surviving coefficients are a short, readable description of what the neuron responds to.

Two λ values are conventionally reported, and the original code saves both. **`lambda_min`**
is the value with the best cross-validated deviance. **`lambda_1se`** is the largest λ whose
CV deviance is within one standard error of the best — a deliberately more conservative,
sparser model, chosen on the argument that the minimum of a noisy CV curve is itself
overfitted. If you report coefficients, `lambda_1se` is the safer choice; if you report
predictive performance, `lambda_min`.

> ### Homework question 4
> **(a)** In the ridge path, coefficients do not reach exactly zero, however large λ becomes.
> Why not, and what changes with an L1 penalty?
>
> **(b)** Set `alpha_l1 = 1.0` (pure lasso) and re-run the penalty sweep on the correlated
> speed-basis columns. Does the set of surviving predictors become less stable across
> penalties?
>
> **(c)** Implement `lambda_1se`: compute the CV deviance per fold, take its standard error
> across folds, and find the largest λ within one SE of the minimum.

---
## Part VII. Nested model comparison — the selectivity measure

Here is the payoff, and the reason this analysis exists.

Fit not one model but four, differing only in which blocks of columns they may use:

| Model | Columns |
|---|---|
| **null** | intercept only |
| **motor** | intercept + speed basis + acceleration |
| **task** | intercept + cue × epoch + outcome |
| **full** | everything |

Then define the **task contribution** as how much held-out deviance the task columns explain
*that the motor columns could not*:

$$\Delta\text{FDE}_{\text{task}} = \text{FDE}_{\text{full}} - \text{FDE}_{\text{motor}}$$

This is the quantity to report when you claim a neuron is task-selective. It is not "does
activity differ across conditions" — the ANOVA already answered that, wrongly. It is "does
knowing the task variables improve prediction of held-out data, given that we already know
what the animal was doing?"

The symmetric quantity $\Delta\text{FDE}_{\text{motor}} = \text{FDE}_{\text{full}} -
\text{FDE}_{\text{task}}$ asks the same of movement.

In [ ]:
MODELS = {"null":  np.array([], int),
          "motor": MOTOR_COLS,
          "task":  TASK_COLS,
          "full":  np.arange(1, X_all.shape[1])}

results = {}
for name, y in Y.items():
    lam = 1.0
    results[name] = {m: cv_fde(y, X_all, cols, lam=lam)[0] for m, cols in MODELS.items()}
    r = results[name]
    r["dFDE_task"]  = r["full"] - r["motor"]
    r["dFDE_motor"] = r["full"] - r["task"]

hdr = f"{'neuron':<12}{'FDE motor':>11}{'FDE task':>10}{'FDE full':>10}{'ΔFDE task':>11}{'ΔFDE motor':>12}"
print(hdr); print("-" * len(hdr))
for name, r in results.items():
    print(f"{name:<12}{r['motor']:>11.4f}{r['task']:>10.4f}{r['full']:>10.4f}"
          f"{r['dFDE_task']:>11.4f}{r['dFDE_motor']:>12.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
names = list(results)
xp = np.arange(len(names))

ax = axes[0]
w = 0.27
for i, (m, col) in enumerate(zip(("motor", "task", "full"), (ORANGE, BLUE, PURPLE))):
    ax.bar(xp + (i - 1) * w, [results[n][m] for n in names], w, color=col, label=m)
ax.axhline(0, color="k", lw=1)
ax.set_xticks(xp); ax.set_xticklabels(names, rotation=20, ha="right")
ax.set(ylabel="Held-out FDE", title="Each model on its own")
ax.legend(fontsize=8)

ax = axes[1]
ax.bar(xp - 0.2, [results[n]["dFDE_task"] for n in names], 0.4, color=BLUE,
       label="ΔFDE task  (task | motor)")
ax.bar(xp + 0.2, [results[n]["dFDE_motor"] for n in names], 0.4, color=ORANGE,
       label="ΔFDE motor (motor | task)")
ax.axhline(0, color="k", lw=1)
ax.set_xticks(xp); ax.set_xticklabels(names, rotation=20, ha="right")
ax.set(ylabel="Δ held-out FDE", title="Unique contribution of each block")
ax.legend(fontsize=8)
fig.tight_layout()

Read the right-hand panel against the table of ground truth from Part IV, and every neuron
lands where it should.

`delay_cue` and `test_epoch` have a large task contribution and essentially no motor
contribution. `mixed` has both. `silent` has neither.

And `speed_only` — the neuron the ANOVA confidently called cue-selective — has a **large
motor contribution and a task contribution near zero**. Once the model already knows the
running speed, the cue columns add nothing to held-out prediction. The apparent selectivity
was speed all along, and the nested comparison says so without being told.

Compare the two panels to see why the comparison has to be nested. In the left panel,
`speed_only` has a clearly positive FDE for the *task* model on its own — task variables do
predict its activity, through the confound. It is only when you ask what task adds *given
motor* that the answer collapses to zero.

> ### Homework question 5
> **(a)** $\Delta\text{FDE}_{\text{task}} + \Delta\text{FDE}_{\text{motor}}$ does not equal
> $\text{FDE}_{\text{full}}$. Explain what the leftover represents, and why it is large
> exactly when the two blocks are correlated. (This is the GLM version of shared variance in
> a partial-$R^2$ decomposition.)
>
> **(b)** Break the task down further: split the task block into cue-selectivity columns
> versus epoch-only columns, and compute a Δ for each. Which neurons are cue-selective as
> opposed to merely task-modulated?
>
> **(c)** Increase the confound (`SPEED_BY_CUE` to ±25 cm/s) and re-run. At what point does
> the nested comparison start to fail, and why? What would you do experimentally?
>
> **(d)** Add a neuron that responds to cue 1 during the delay — the same cue on which the
> animal runs fastest. Is its task contribution still recovered? What does this tell you
> about which selectivity is hardest to establish?

In [ ]:
# Does the full model recover the coefficients we planted?
fig, axes = plt.subplots(1, len(Y), figsize=(15, 3.4), sharey=True)
for ax, (name, y) in zip(axes, Y.items()):
    b_hat  = poisson_glm_fit(X_all, y, lam=1.0)
    b_true = NEURONS[name]
    idx = np.arange(1, len(labels_all))
    ax.axhline(0, color="k", lw=0.8)
    ax.plot(idx, b_true[idx], "o", ms=5, color="k", label="true", zorder=3)
    ax.plot(idx, b_hat[idx],  "x", ms=6, color=RED, mew=1.6, label="fitted")
    ax.axvspan(0.5, 17.5, color=BLUE, alpha=0.08)
    ax.axvspan(17.5, len(labels_all) - 0.5, color=ORANGE, alpha=0.10)
    ax.set(xlabel="Predictor index", title=name)
axes[0].set_ylabel("Coefficient")
axes[0].legend(fontsize=8)
fig.suptitle("Recovered vs planted coefficients   "
             "(blue band = task block, orange = motor block)", y=1.06)
fig.tight_layout()

The fitted coefficients track the planted ones closely for the neurons whose drive is in the
model. Look at `speed_only`, though: its fitted **task** coefficients are not all zero, even
though its true ones are. The confound leaves a residual imprint on individual coefficients
even when the nested comparison correctly reports no task contribution.

That is a useful warning. Reading selectivity off individual coefficients is fragile when
predictors are correlated — a single large $\beta$ for `cue1_in_delay` does not establish cue
selectivity. The held-out, model-level comparison is the robust statement; the coefficients
are for interpretation *after* that test passes.

---
## Part VIII. Is it significant? Trial shuffling

$\Delta\text{FDE}_{\text{task}} = 0.02$ — is that a real effect or noise? As in the
classification tutorial, the answer comes from a null distribution generated by breaking the
thing you care about while preserving everything else.

The right shuffle here is **circular trial shuffling of the task labels**: reassign the cue
identities to different trials, keeping each trial's epoch structure and speed trace intact.
That destroys the cue–activity relationship while preserving the temporal autocorrelation of
the activity, the epoch structure, and the speed statistics — all of which would inflate a
naive null.

In [ ]:
def shuffled_task_columns(rng):
    '''Rebuild the cue x epoch block with cue identities reassigned across trials.'''
    cue_shuf = cue[rng.permutation(n_trials)]
    cue_f = cue_shuf[trial_id]
    out = X_all.copy()
    j = 1
    for ep in ["sample", "delay", "test", "reward"]:
        for c in (1, 2, 3, 4):
            out[:, j] = ((epoch == ep) & (cue_f == c)).astype(float)
            j += 1
    return out

n_shuf = 100
null_dist = {n: np.empty(n_shuf) for n in Y}
for si in range(n_shuf):
    Xs = shuffled_task_columns(rng)
    for name, y in Y.items():
        f_full, _  = cv_fde(y, Xs, np.arange(1, Xs.shape[1]), lam=1.0)
        f_motor, _ = cv_fde(y, Xs, MOTOR_COLS, lam=1.0)
        null_dist[name][si] = f_full - f_motor

print(f"{'neuron':<12}{'observed':>10}{'null mean':>11}{'null SD':>9}{'p':>8}")
print("-" * 50)
pvals = {}
for name in Y:
    obs = results[name]["dFDE_task"]
    nd = null_dist[name]
    pvals[name] = (1 + np.sum(nd >= obs)) / (1 + n_shuf)
    print(f"{name:<12}{obs:>10.4f}{nd.mean():>11.4f}{nd.std():>9.4f}{pvals[name]:>8.3f}")

In [ ]:
fig, axes = plt.subplots(1, len(Y), figsize=(15, 2.9), sharey=True)
for ax, name in zip(axes, Y):
    ax.hist(null_dist[name], bins=12, color="0.7")
    ax.axvline(results[name]["dFDE_task"], color=RED, lw=2)
    ax.set_title(f"{name}\np = {pvals[name]:.3f}", fontsize=9)
    ax.set_xlabel("ΔFDE task")
axes[0].set_ylabel("Shuffles")
fig.suptitle("Observed task contribution (red) against the trial-shuffled null", y=1.10)
fig.tight_layout()

Three results, and the middle one is the most interesting.

`delay_cue` and `mixed` sit far outside their null distributions — correctly called
cue-selective. `speed_only` and `silent` sit inside theirs — correctly called not
cue-selective.

**`test_epoch` also sits inside its null, and that is the right answer.** Shuffling *cue
identity* across trials leaves its epoch structure completely intact, and epoch structure is
all this neuron has. So the test reports, correctly, that it is not **cue**-selective — even
though Part VII showed it has a large task contribution. The two measures answer different
questions: ΔFDE asks "do task variables matter at all?", while this shuffle asks "does cue
identity specifically matter?". A neuron can pass the first and fail the second, and
`test_epoch` is exactly that neuron. This is the distinction homework 2(a) asked you to
articulate, now made quantitative.

Note also that the null distributions are centred near zero but **not exactly at zero**, and
their width differs markedly between neurons. That is why a fixed threshold on ΔFDE ("call it
selective if ΔFDE > 0.01") is worse than a per-neuron shuffle test: how much ΔFDE a neuron can
produce by chance depends on its firing rate and how strongly it is speed-modulated.

> ### Homework question 6
> **(a)** We shuffled cue identity across trials but kept epoch structure. Design a shuffle
> that tests epoch modulation instead, and say what it must preserve.
>
> **(b)** With 30 shuffles the smallest achievable p-value is about 0.03. How many shuffles
> do you need to support a claim at p < 0.001, and what does that cost in compute?
>
> **(c)** Across a real population of a thousand neurons you would run this test a thousand
> times. Look up the false discovery rate and Benjamini-Hochberg correction, and apply it to
> the p-values above.

### Where splitting by trial really matters

Part V found that frame-wise versus trial-wise *cross-validation* barely changed the point
estimate. Shuffling is a different story. Compare shuffling cue labels **per trial** —
preserving the fact that the cue is constant within a trial — against shuffling them **per
frame**, which scrambles cue identity within trials as well.

In [ ]:
def shuffled_per_frame(rng):
    '''WRONG null: reassign cue identity independently on every frame.'''
    cue_f = cue[rng.integers(0, n_trials, n_frames)]
    out = X_all.copy()
    j = 1
    for ep in ["sample", "delay", "test", "reward"]:
        for c in (1, 2, 3, 4):
            out[:, j] = ((epoch == ep) & (cue_f == c)).astype(float)
            j += 1
    return out

n_cmp = 40
null_frame = {n: np.empty(n_cmp) for n in Y}
for si in range(n_cmp):
    Xs = shuffled_per_frame(rng)
    for name, y in Y.items():
        f_full, _  = cv_fde(y, Xs, np.arange(1, Xs.shape[1]), lam=1.0)
        f_motor, _ = cv_fde(y, Xs, MOTOR_COLS, lam=1.0)
        null_frame[name][si] = f_full - f_motor

print(f"{'neuron':<12}{'null SD (trial)':>17}{'null SD (frame)':>17}{'ratio':>9}")
print("-" * 57)
for name in Y:
    a, b = null_dist[name].std(), null_frame[name].std()
    print(f"{name:<12}{a:>17.5f}{b:>17.5f}{a/max(b,1e-12):>8.1f}x")

fig, ax = plt.subplots(figsize=(6.4, 4.2))
for name, col in zip(Y, (BLUE, GREEN, RED, PURPLE, GREY)):
    ax.scatter(null_dist[name].std(), null_frame[name].std(), s=80, color=col, label=name)
lim = [1e-5, 1e-1]
ax.plot(lim, lim, "k:", lw=1)
ax.set(xscale="log", yscale="log", xlim=lim, ylim=lim,
       xlabel="Null SD, shuffled by TRIAL", ylabel="Null SD, shuffled by FRAME",
       title="Frame-wise shuffling understates the null spread")
ax.legend(fontsize=8)
fig.tight_layout()

The inflation is not uniform, and the pattern tells you the mechanism. It is large exactly for
the neurons that have genuine trial-level cue structure — 19x for `delay_cue`, 5x for `mixed`
— and negligible (about 1.1x) for the three that do not.

Here is why. When you shuffle whole trials, a shuffled assignment can by chance partially
align with the true cue pattern: with 240 trials and 4 cues, some reshuffles land closer to
the truth than others, and that produces real spread in the null. That spread is exactly the
uncertainty you need to account for. Shuffling frames independently can never produce a
coherent trial-level pattern, so the null collapses to a spike near zero — it is a
distribution over sessions that could not occur. Measured against it, `delay_cue` sits ~700
standard deviations from the mean rather than ~39.

This is the concrete version of the warning in Part V. Frame-wise resampling did not
noticeably bias the point *estimate*, but it inflates significance by more than an order of
magnitude for precisely the neurons whose significance you care about — and significance is
what gets reported. **Resample at the level at which your data are independent**, which here
means the trial.

---
## Summary

1. **Encoding, not decoding.** A GLM predicts activity from task and behavioural variables.
   That is the direction in which selectivity questions live.

2. **Three modelling choices define a GLM**: the linear predictor, the link (log, giving
   multiplicative effects and positive rates), and the noise family (Poisson for counts).
   Deviance generalizes the residual sum of squares, and held-out **fraction of deviance
   explained** generalizes $R^2$.

3. **The design matrix is the analysis.** Cue × epoch indicators make selectivity
   epoch-specific; a raised-cosine basis lets speed act nonlinearly; lags absorb the slow
   indicator. An ITI period keeps the intercept identifiable.

4. **Split by trial, never by frame.** Neighbouring frames are nearly the same observation.
   Frame-wise splits inflate FDE for a neuron with no structure at all.

5. **Regularize.** Elastic net with $\alpha \approx 0.95$ gives sparse, readable models
   without lasso's instability under correlated predictors.

6. **Nested model comparison is the selectivity measure.** $\Delta\text{FDE}_{\text{task}} =
   \text{FDE}_{\text{full}} - \text{FDE}_{\text{motor}}$ asks whether task variables predict
   held-out activity *given* what the animal was doing. A one-way ANOVA on epoch-averaged
   activity called our pure speed neuron cue-selective at $p < 10^{-30}$; the nested
   comparison correctly gave it a task contribution of essentially zero.

7. **Significance by shuffling the thing you care about**, preserving everything else.

The general lesson outlives the method. In a behaving animal, every variable is correlated
with every other, and "this neuron responds to X" is a claim about a *conditional*
relationship. The GLM's real contribution is not that it fits curves — it is that it forces
you to name the alternatives explicitly and then measures whether your variable survives
them.

### Further reading

- Nelder & Wedderburn (1972). Generalized linear models. *J. R. Statist. Soc. A* **135**,
  370–384.
- Truccolo, Eden, Fellows, Donoghue & Brown (2005). A point process framework for relating
  neural spiking activity to spiking history, neural ensemble, and extrinsic covariate
  effects. *Journal of Neurophysiology* **93**, 1074–1089.
- Pillow et al. (2008). Spatio-temporal correlations and visual signalling in a complete
  neuronal population. *Nature* **454**, 995–999.
- Friedman, Hastie & Tibshirani (2010). Regularization paths for generalized linear models
  via coordinate descent. *Journal of Statistical Software* **33**, 1–22. (The glmnet paper.)
- Musall, Kaufman, Juavinett, Gluf & Churchland (2019). Single-trial neural dynamics are
  dominated by richly varied movements. *Nature Neuroscience* **22**, 1677–1686. (Why the
  movement confound in Part IV is not a hypothetical.)
- Hastie, Tibshirani & Friedman (2009). *The Elements of Statistical Learning*, chapter 3.